# Downscaling job runner

This notebook is the "worker" half of the Drive-relay setup: run every cell below, authorize Drive access when prompted, and leave this tab open. It polls `MyDrive/climate-downscaling-jobs/` every 10 seconds for jobs submitted from the companion web app (`index.html`), runs the pipeline for each one, and writes logs/status back to the same folder so the web app can display them.

**Before running:** set your GEE project ID is passed per-job from the web app's params JSON, but Earth Engine still needs to be authorized once in this runtime — the setup cell below handles that interactively.

In [ ]:
# 1) Clone the pipeline and install dependencies
!git clone -q https://github.com/awan-geospatial1/climate-downscaling.git
%cd climate-downscaling
!pip install -q -r requirements.txt

In [ ]:
# 2) Mount Drive (this is the shared filesystem the web app writes into via the Drive API)
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 3) Authorize Earth Engine once for this runtime, so run_pipeline's ee.Initialize() succeeds
#    without re-prompting for every job.
import ee
EE_PROJECT_FOR_AUTH = "your-gee-project-id"  # any project you have EE access on; each job can still use its own gee_project_id
try:
    ee.Initialize(project=EE_PROJECT_FOR_AUTH)
    print("EE already authorized.")
except Exception:
    ee.Authenticate()
    ee.Initialize(project=EE_PROJECT_FOR_AUTH)
    print("EE authorized.")

In [ ]:
# 4) Poll loop: watches for new request.json files, runs run_pipeline, streams logs
import os, json, time, traceback, io, contextlib, sys, zipfile, glob
from datetime import datetime, timezone
from main import run_pipeline

JOBS_DIR = "/content/drive/MyDrive/climate-downscaling-jobs"
POLL_SECONDS = 10
os.makedirs(JOBS_DIR, exist_ok=True)

def now():
    return datetime.now(timezone.utc).isoformat()

def append_log(job_dir, msg):
    with open(os.path.join(job_dir, "log.txt"), "a") as f:
        f.write(f"[{now()}] {msg}\n")

def write_status(job_dir, status, extra=None):
    d = {"status": status, "updated": now()}
    if extra:
        d.update(extra)
    with open(os.path.join(job_dir, "status.json"), "w") as f:
        json.dump(d, f)

class LiveLogWriter(io.TextIOBase):
    """Redirect stdout from run_pipeline straight into log.txt, line by line."""
    def __init__(self, path):
        self.path = path
    def write(self, s):
        if s.strip():
            with open(self.path, "a") as f:
                f.write(f"[{now()}] {s.rstrip()}\n")
        return len(s)

def resolve_aoi_path(job_dir):
    """
    index.html always uploads the AOI as aoi.zip (a zipped shapefile --
    .shp/.shx/.dbf/.prj), never a raw .geojson. This used to hardcode
    aoi.geojson unconditionally, which never existed, so every job failed
    at the exact same load_shapefile() call regardless of what was
    actually submitted. Extract the zip and find the .shp inside instead.
    Falls back to aoi.geojson if that's what's actually there, in case a
    job was staged some other way.
    """
    geojson_path = os.path.join(job_dir, "aoi.geojson")
    zip_path = os.path.join(job_dir, "aoi.zip")

    if os.path.isfile(geojson_path):
        return geojson_path

    if os.path.isfile(zip_path):
        extract_dir = os.path.join(job_dir, "aoi_extracted")
        os.makedirs(extract_dir, exist_ok=True)
        with zipfile.ZipFile(zip_path) as zf:
            zf.extractall(extract_dir)
        matches = glob.glob(os.path.join(extract_dir, "**", "*.shp"), recursive=True)
        if not matches:
            raise FileNotFoundError(
                f"aoi.zip was found but contained no .shp file (job: {job_dir})")
        return matches[0]

    raise FileNotFoundError(
        f"Neither aoi.geojson nor aoi.zip found in {job_dir} -- "
        f"the upload may not have finished before this job was picked up.")


def process_job(job_dir, job_name):
    req_path = os.path.join(job_dir, "request.json")
    log_path = os.path.join(job_dir, "log.txt")

    write_status(job_dir, "running")
    append_log(job_dir, f"Starting job {job_name}")
    try:
        aoi_path = resolve_aoi_path(job_dir)
        append_log(job_dir, f"Using AOI file: {aoi_path}")

        with open(req_path) as f:
            params = json.load(f)
        params["shapefile_path"] = aoi_path
        out_dir = os.path.join(job_dir, "outputs")
        os.makedirs(out_dir, exist_ok=True)
        params["output_dir"] = out_dir

        append_log(job_dir, f"Params: {json.dumps(params)}")
        with contextlib.redirect_stdout(LiveLogWriter(log_path)):
            run_pipeline(params)

        write_status(job_dir, "done")
        append_log(job_dir, "Job complete. Outputs in outputs/.")
    except Exception as e:
        append_log(job_dir, f"ERROR: {e}\n{traceback.format_exc()}")
        write_status(job_dir, "error", {"error": str(e)})

print(f"Polling {JOBS_DIR} every {POLL_SECONDS}s. Leave this cell running.")
while True:
    try:
        for job_name in sorted(os.listdir(JOBS_DIR)):
            job_dir = os.path.join(JOBS_DIR, job_name)
            req_path = os.path.join(job_dir, "request.json")
            status_path = os.path.join(job_dir, "status.json")
            if not os.path.isfile(req_path):
                continue
            if os.path.isfile(status_path):
                with open(status_path) as f:
                    if json.load(f).get("status") in ("running", "done", "error"):
                        continue
            process_job(job_dir, job_name)
    except Exception as loop_err:
        print(f"[{now()}] poll loop error: {loop_err}")
    time.sleep(POLL_SECONDS)